In [7]:
import os
import glob
import pandas as pd
import numpy as np
import time
from sentence_transformers import SentenceTransformer, util
import torch
from sklearn.decomposition import PCA
import pickle
import ast


In [2]:
# Load the model
model = SentenceTransformer('all-mpnet-base-v2')

# Get class names from unseen folder structure (you can adjust path as needed)
image_class = [
    "chimpanzee",
    "giant panda",
    "hippopotamus",
    "humpback whale",
    "leopard",
    "persian cat",
    "pig",
    "raccoon",
    "rat",
    "seal"
]

# Index mapping
idx_to_unseen_class = {k: v for k, v in enumerate(image_class)}
words = list(idx_to_unseen_class.values())

# Encode class labels
embeddings = model.encode(words, convert_to_tensor=True)
print(embeddings)
print(embeddings.shape)


tensor([[ 0.0196,  0.0364, -0.0098,  ...,  0.0088,  0.0120, -0.0178],
        [ 0.0595,  0.0703,  0.0081,  ...,  0.0154,  0.0058,  0.0157],
        [ 0.0365,  0.0571, -0.0143,  ...,  0.0391,  0.0143, -0.0131],
        ...,
        [ 0.0440,  0.0233,  0.0111,  ...,  0.0058,  0.0182, -0.0209],
        [-0.0027,  0.0249, -0.0028,  ..., -0.0043,  0.0178, -0.0261],
        [-0.0148,  0.0676, -0.0262,  ..., -0.0004, -0.0211, -0.0234]])
torch.Size([10, 768])


In [3]:
# Convert embeddings to numpy array for PCA
embeddings_np = embeddings.cpu().numpy()
# embeddings_np.shape

# Apply PCA to reduce from 768 to 16 dimensions
pca = PCA(n_components=10)
embeddings_reduced = pca.fit_transform(embeddings_np)

# Convert back to torch tensor if needed
embeddings_reduced_tensor = torch.from_numpy(embeddings_reduced)

print(f"Original embeddings shape: {embeddings.shape}")
print(f"Reduced embeddings shape: {embeddings_reduced_tensor.shape}")
# print(f"Explained variance ratio: {np.sum(pca.explained_variance_ratio_):.4f}")

Original embeddings shape: torch.Size([10, 768])
Reduced embeddings shape: torch.Size([10, 10])


In [4]:
# Save the PCA model to a file
with open('pca_model.pkl', 'wb') as f:
    pickle.dump(pca, f)
    
print("PCA model saved to 'pca_model.pkl'")

# Optionally save the reduced class embeddings for reference
np.save('class_embeddings_reduced.npy', embeddings_reduced)
print("Reduced class embeddings saved to 'class_embeddings_reduced.npy'")

PCA model saved to 'pca_model.pkl'
Reduced class embeddings saved to 'class_embeddings_reduced.npy'


In [6]:
# Load the saved PCA model
with open('pca_model.pkl', 'rb') as f:
    pca = pickle.load(f)

# Path to CSV file
csv_file_path = "prediction_results_Resnet_GLCM_AWA2_chimpanzee.csv"
df = pd.read_csv(csv_file_path)

# Clean pred_image_label column
df['cleaned_pred_label'] = df['pred_image_label'].str.replace('_', ' ', regex=False)

# Timing start
start_time = time.time()

# Similarity processing
similar_wds = []

for _, row in df.iterrows():
    query_label = row['cleaned_pred_label']

    # Encode predicted label
    query_embedding = model.encode(query_label, convert_to_tensor=True)
  
    # Convert to numpy and apply PCA transformation
    query_embedding_np = query_embedding.cpu().numpy()
    
    # Reshape if needed (PCA expects 2D array)
    if query_embedding_np.ndim == 1:
        query_embedding_np = query_embedding_np.reshape(1, -1)
    
    # Apply the same PCA transformation
    query_embedding_reduced = pca.transform(query_embedding_np)
    
    # Convert back to tensor
    query_embedding_reduced_tensor = torch.from_numpy(query_embedding_reduced)
    
    # Compute cosine similarity
    # cosine_scores = util.cos_sim(query_embedding, embeddings)
    cosine_scores = util.cos_sim(query_embedding_reduced_tensor, embeddings_reduced_tensor)

    # Get top-5 most similar class names
    top_results = torch.topk(cosine_scores[0], k=5)

    top_labels = [words[idx] for idx in top_results.indices]
    top_scores = [round(float(cosine_scores[0][idx]) * 100, 2) for idx in top_results.indices]

    # Store as a tuple in a new column
    similar_wds.append((top_labels, top_scores))

# Timing end
end_time = time.time()
total_time = end_time - start_time
time_per_image = total_time / len(df)

print(f"Total images processed: {len(df)}")
print(f"Total processing time: {total_time:.4f} seconds")
print(f"Average time per image: {time_per_image:.4f} seconds")

# Add new column with results
df['top5_similar_labels_with_scores'] = similar_wds

# Save the updated CSV
output_csv_path = "prediction_results_Resnet_GLCM_AWA2_chimpanzee_with_Transfomer_similarity.csv"
df.to_csv(output_csv_path, index=False)
print(f"✅ Done! Saved to '{output_csv_path}'")


Total images processed: 728
Total processing time: 70.1999 seconds
Average time per image: 0.0964 seconds
✅ Done! Saved to 'prediction_results_Resnet_GLCM_AWA2_chimpanzee_with_Transfomer_similarity.csv'


In [8]:
# Load your CSV
df = pd.read_csv("prediction_results_Resnet_GLCM_AWA2_chimpanzee_with_Transfomer_similarity.csv")  # Change if needed
print(df.columns.tolist())

# Clean actual labels
df['image_label'] = df['image_label'].str.replace("_", " ").str.lower().str.strip()

# Parse top5 labels with scores
def extract_labels(top5_str):
    try:
        parsed = ast.literal_eval(top5_str)
        return [label.lower().strip() for label in parsed[0]]  # First part is labels
    except:
        return []

df['top_5_predicted'] = df['top5_similar_labels_with_scores'].apply(extract_labels)

# Compute hit (True if actual label is in top-5)
df['top5_hit'] = df.apply(lambda row: row['image_label'] in row['top_5_predicted'], axis=1)

# Compute per-class accuracy
accuracy_per_class = df.groupby('image_label')['top5_hit'].agg(['sum', 'count']).reset_index()
accuracy_per_class['top5_accuracy (%)'] = round((accuracy_per_class['sum'] / accuracy_per_class['count']) * 100, 2)
accuracy_per_class.rename(columns={'sum': 'correct', 'count': 'total'}, inplace=True)

# Average accuracy
average_top5_accuracy = round(accuracy_per_class['top5_accuracy (%)'].mean(), 2)

# Output
print("\nTop-5 Accuracy per Class:")
print(accuracy_per_class)

print(f"\nAverage Top-5 Accuracy: {average_top5_accuracy:.2f}%")

# Optional: Save
accuracy_per_class.to_csv("top5_accuracy_summary.csv", index=False)


['image_path', 'image_label', 'pred_image_label', 'act_label', 'pred_label', 'cleaned_pred_label', 'top5_similar_labels_with_scores']

Top-5 Accuracy per Class:
  image_label  correct  total  top5_accuracy (%)
0  chimpanzee      672    728              92.31

Average Top-5 Accuracy: 92.31%


# giant_panda

In [9]:
# Load the saved PCA model
with open('pca_model.pkl', 'rb') as f:
    pca = pickle.load(f)

# Path to CSV file
csv_file_path = "prediction_results_Resnet_GLCM_AWA2_giant_panda.csv"
df = pd.read_csv(csv_file_path)

# Clean pred_image_label column
df['cleaned_pred_label'] = df['pred_image_label'].str.replace('_', ' ', regex=False)

# Timing start
start_time = time.time()

# Similarity processing
similar_wds = []

for _, row in df.iterrows():
    query_label = row['cleaned_pred_label']

    # Encode predicted label
    query_embedding = model.encode(query_label, convert_to_tensor=True)
  
    # Convert to numpy and apply PCA transformation
    query_embedding_np = query_embedding.cpu().numpy()
    
    # Reshape if needed (PCA expects 2D array)
    if query_embedding_np.ndim == 1:
        query_embedding_np = query_embedding_np.reshape(1, -1)
    
    # Apply the same PCA transformation
    query_embedding_reduced = pca.transform(query_embedding_np)
    
    # Convert back to tensor
    query_embedding_reduced_tensor = torch.from_numpy(query_embedding_reduced)
    
    # Compute cosine similarity
    # cosine_scores = util.cos_sim(query_embedding, embeddings)
    cosine_scores = util.cos_sim(query_embedding_reduced_tensor, embeddings_reduced_tensor)

    # Get top-5 most similar class names
    top_results = torch.topk(cosine_scores[0], k=5)

    top_labels = [words[idx] for idx in top_results.indices]
    top_scores = [round(float(cosine_scores[0][idx]) * 100, 2) for idx in top_results.indices]

    # Store as a tuple in a new column
    similar_wds.append((top_labels, top_scores))

# Timing end
end_time = time.time()
total_time = end_time - start_time
time_per_image = total_time / len(df)

print(f"Total images processed: {len(df)}")
print(f"Total processing time: {total_time:.4f} seconds")
print(f"Average time per image: {time_per_image:.4f} seconds")

# Add new column with results
df['top5_similar_labels_with_scores'] = similar_wds

# Save the updated CSV
output_csv_path = "prediction_results_Resnet_GLCM_AWA2_giant_panda_with_Transfomer_similarity.csv"
df.to_csv(output_csv_path, index=False)
print(f"✅ Done! Saved to '{output_csv_path}'")


Total images processed: 874
Total processing time: 95.7529 seconds
Average time per image: 0.1096 seconds
✅ Done! Saved to 'prediction_results_Resnet_GLCM_AWA2_giant_panda_with_Transfomer_similarity.csv'


In [10]:
# Load your CSV
df = pd.read_csv("prediction_results_Resnet_GLCM_AWA2_giant_panda_with_Transfomer_similarity.csv")  # Change if needed
print(df.columns.tolist())

# Clean actual labels
df['image_label'] = df['image_label'].str.replace("_", " ").str.lower().str.strip()

# Parse top5 labels with scores
def extract_labels(top5_str):
    try:
        parsed = ast.literal_eval(top5_str)
        return [label.lower().strip() for label in parsed[0]]  # First part is labels
    except:
        return []

df['top_5_predicted'] = df['top5_similar_labels_with_scores'].apply(extract_labels)

# Compute hit (True if actual label is in top-5)
df['top5_hit'] = df.apply(lambda row: row['image_label'] in row['top_5_predicted'], axis=1)

# Compute per-class accuracy
accuracy_per_class = df.groupby('image_label')['top5_hit'].agg(['sum', 'count']).reset_index()
accuracy_per_class['top5_accuracy (%)'] = round((accuracy_per_class['sum'] / accuracy_per_class['count']) * 100, 2)
accuracy_per_class.rename(columns={'sum': 'correct', 'count': 'total'}, inplace=True)

# Average accuracy
average_top5_accuracy = round(accuracy_per_class['top5_accuracy (%)'].mean(), 2)

# Output
print("\nTop-5 Accuracy per Class:")
print(accuracy_per_class)

print(f"\nAverage Top-5 Accuracy: {average_top5_accuracy:.2f}%")

# Optional: Save
accuracy_per_class.to_csv("top5_accuracy_summary.csv", index=False)


['image_path', 'image_label', 'pred_image_label', 'act_label', 'pred_label', 'cleaned_pred_label', 'top5_similar_labels_with_scores']

Top-5 Accuracy per Class:
   image_label  correct  total  top5_accuracy (%)
0  giant panda      802    874              91.76

Average Top-5 Accuracy: 91.76%


In [11]:
def pca_result(csv_file_path, output_csv_path):
    # Load the saved PCA model
    with open('pca_model.pkl', 'rb') as f:
        pca = pickle.load(f)

    # Path to CSV file
    # csv_file_path = "prediction_results_Resnet_GLCM_AWA2_giant_panda.csv"
    df = pd.read_csv(csv_file_path)

    # Clean pred_image_label column
    df['cleaned_pred_label'] = df['pred_image_label'].str.replace('_', ' ', regex=False)

    # Timing start
    start_time = time.time()

    # Similarity processing
    similar_wds = []

    for _, row in df.iterrows():
        query_label = row['cleaned_pred_label']

        # Encode predicted label
        query_embedding = model.encode(query_label, convert_to_tensor=True)

        # Convert to numpy and apply PCA transformation
        query_embedding_np = query_embedding.cpu().numpy()

        # Reshape if needed (PCA expects 2D array)
        if query_embedding_np.ndim == 1:
            query_embedding_np = query_embedding_np.reshape(1, -1)

        # Apply the same PCA transformation
        query_embedding_reduced = pca.transform(query_embedding_np)

        # Convert back to tensor
        query_embedding_reduced_tensor = torch.from_numpy(query_embedding_reduced)

        # Compute cosine similarity
        # cosine_scores = util.cos_sim(query_embedding, embeddings)
        cosine_scores = util.cos_sim(query_embedding_reduced_tensor, embeddings_reduced_tensor)

        # Get top-5 most similar class names
        top_results = torch.topk(cosine_scores[0], k=5)

        top_labels = [words[idx] for idx in top_results.indices]
        top_scores = [round(float(cosine_scores[0][idx]) * 100, 2) for idx in top_results.indices]

        # Store as a tuple in a new column
        similar_wds.append((top_labels, top_scores))

    # Timing end
    end_time = time.time()
    total_time = end_time - start_time
    time_per_image = total_time / len(df)

    print(f"Total images processed: {len(df)}")
    print(f"Total processing time: {total_time:.4f} seconds")
    print(f"Average time per image: {time_per_image:.4f} seconds")

    # Add new column with results
    df['top5_similar_labels_with_scores'] = similar_wds

    # Save the updated CSV
    # output_csv_path = "prediction_results_Resnet_GLCM_AWA2_giant_panda_with_Transfomer_similarity.csv"
    df.to_csv(output_csv_path, index=False)
    print(f"✅ Done! Saved to '{output_csv_path}'")
    
    # Load your CSV
    df = pd.read_csv(output_csv_path)  # Change if needed
    print(df.columns.tolist())

    # Clean actual labels
    df['image_label'] = df['image_label'].str.replace("_", " ").str.lower().str.strip()

    # Parse top5 labels with scores
    def extract_labels(top5_str):
        try:
            parsed = ast.literal_eval(top5_str)
            return [label.lower().strip() for label in parsed[0]]  # First part is labels
        except:
            return []

    df['top_5_predicted'] = df['top5_similar_labels_with_scores'].apply(extract_labels)

    # Compute hit (True if actual label is in top-5)
    df['top5_hit'] = df.apply(lambda row: row['image_label'] in row['top_5_predicted'], axis=1)

    # Compute per-class accuracy
    accuracy_per_class = df.groupby('image_label')['top5_hit'].agg(['sum', 'count']).reset_index()
    accuracy_per_class['top5_accuracy (%)'] = round((accuracy_per_class['sum'] / accuracy_per_class['count']) * 100, 2)
    accuracy_per_class.rename(columns={'sum': 'correct', 'count': 'total'}, inplace=True)

    # Average accuracy
    average_top5_accuracy = round(accuracy_per_class['top5_accuracy (%)'].mean(), 2)

    # Output
    print("\nTop-5 Accuracy per Class:")
    print(accuracy_per_class)

    print(f"\nAverage Top-5 Accuracy: {average_top5_accuracy:.2f}%")

    # Optional: Save
    accuracy_per_class.to_csv("top5_accuracy_summary.csv", index=False)


# hippopotamus

In [12]:
pca_result("prediction_results_Resnet_GLCM_AWA2_hippopotamus.csv", 
           "prediction_results_Resnet_GLCM_AWA2_hippopotamus_with_Transfomer_similarity.csv")

Total images processed: 684
Total processing time: 48.1982 seconds
Average time per image: 0.0705 seconds
✅ Done! Saved to 'prediction_results_Resnet_GLCM_AWA2_hippopotamus_with_Transfomer_similarity.csv'
['image_path', 'image_label', 'pred_image_label', 'act_label', 'pred_label', 'cleaned_pred_label', 'top5_similar_labels_with_scores']

Top-5 Accuracy per Class:
    image_label  correct  total  top5_accuracy (%)
0  hippopotamus      608    684              88.89

Average Top-5 Accuracy: 88.89%


# humpback_whale

In [13]:
pca_result("prediction_results_Resnet_GLCM_AWA2_humpback_whale.csv", 
           "prediction_results_Resnet_GLCM_AWA2_humpback_whale_with_Transfomer_similarity.csv")

Total images processed: 709
Total processing time: 48.5553 seconds
Average time per image: 0.0685 seconds
✅ Done! Saved to 'prediction_results_Resnet_GLCM_AWA2_humpback_whale_with_Transfomer_similarity.csv'
['image_path', 'image_label', 'pred_image_label', 'act_label', 'pred_label', 'cleaned_pred_label', 'top5_similar_labels_with_scores']

Top-5 Accuracy per Class:
      image_label  correct  total  top5_accuracy (%)
0  humpback whale      709    709              100.0

Average Top-5 Accuracy: 100.00%


# leopard

In [14]:
pca_result("prediction_results_Resnet_GLCM_AWA2_leopard", 
           "prediction_results_Resnet_GLCM_AWA2_leopard_with_Transfomer_similarity.csv")

Total images processed: 720
Total processing time: 49.8205 seconds
Average time per image: 0.0692 seconds
✅ Done! Saved to 'prediction_results_Resnet_GLCM_AWA2_leopard_with_Transfomer_similarity.csv'
['image_path', 'image_label', 'pred_image_label', 'act_label', 'pred_label', 'cleaned_pred_label', 'top5_similar_labels_with_scores']

Top-5 Accuracy per Class:
  image_label  correct  total  top5_accuracy (%)
0     leopard      720    720              100.0

Average Top-5 Accuracy: 100.00%


# percian_cat

In [15]:
pca_result("prediction_results_Resnet_GLCM_AWA2_percian_cat.csv", 
           "prediction_results_Resnet_GLCM_AWA2_percian_cat_with_Transfomer_similarity.csv")

Total images processed: 747
Total processing time: 52.3237 seconds
Average time per image: 0.0700 seconds
✅ Done! Saved to 'prediction_results_Resnet_GLCM_AWA2_percian_cat_with_Transfomer_similarity.csv'
['image_path', 'image_label', 'pred_image_label', 'act_label', 'pred_label', 'cleaned_pred_label', 'top5_similar_labels_with_scores']

Top-5 Accuracy per Class:
   image_label  correct  total  top5_accuracy (%)
0  persian cat      710    747              95.05

Average Top-5 Accuracy: 95.05%


# pig

In [16]:
pca_result("prediction_results_Resnet_GLCM_AWA2_pig.csv", 
           "prediction_results_Resnet_GLCM_AWA2_pig_with_Transfomer_similarity.csv")

Total images processed: 713
Total processing time: 50.0672 seconds
Average time per image: 0.0702 seconds
✅ Done! Saved to 'prediction_results_Resnet_GLCM_AWA2_pig_with_Transfomer_similarity.csv'
['image_path', 'image_label', 'pred_image_label', 'act_label', 'pred_label', 'cleaned_pred_label', 'top5_similar_labels_with_scores']

Top-5 Accuracy per Class:
  image_label  correct  total  top5_accuracy (%)
0         pig      523    713              73.35

Average Top-5 Accuracy: 73.35%


# raccoon

In [17]:
pca_result("prediction_results_Resnet_GLCM_AWA2_raccoon", 
           "prediction_results_Resnet_GLCM_AWA2_raccoon_with_Transfomer_similarity.csv")

Total images processed: 512
Total processing time: 37.7672 seconds
Average time per image: 0.0738 seconds
✅ Done! Saved to 'prediction_results_Resnet_GLCM_AWA2_raccoon_with_Transfomer_similarity.csv'
['image_path', 'image_label', 'pred_image_label', 'act_label', 'pred_label', 'cleaned_pred_label', 'top5_similar_labels_with_scores']

Top-5 Accuracy per Class:
  image_label  correct  total  top5_accuracy (%)
0     raccoon      439    512              85.74

Average Top-5 Accuracy: 85.74%


# rat

In [18]:
pca_result("prediction_results_Resnet_GLCM_AWA2_rat", 
           "prediction_results_Resnet_GLCM_AWA2_rat_with_Transfomer_similarity.csv")

Total images processed: 310
Total processing time: 21.5040 seconds
Average time per image: 0.0694 seconds
✅ Done! Saved to 'prediction_results_Resnet_GLCM_AWA2_rat_with_Transfomer_similarity.csv'
['image_path', 'image_label', 'pred_image_label', 'act_label', 'pred_label', 'cleaned_pred_label', 'top5_similar_labels_with_scores']

Top-5 Accuracy per Class:
  image_label  correct  total  top5_accuracy (%)
0         rat      277    310              89.35

Average Top-5 Accuracy: 89.35%


# seal

In [19]:
pca_result("prediction_results_Resnet_GLCM_AWA2_seal", 
           "prediction_results_Resnet_GLCM_AWA2_seal_with_Transfomer_similarity.csv")

Total images processed: 988
Total processing time: 64.0480 seconds
Average time per image: 0.0648 seconds
✅ Done! Saved to 'prediction_results_Resnet_GLCM_AWA2_seal_with_Transfomer_similarity.csv'
['image_path', 'image_label', 'pred_image_label', 'act_label', 'pred_label', 'cleaned_pred_label', 'top5_similar_labels_with_scores']

Top-5 Accuracy per Class:
  image_label  correct  total  top5_accuracy (%)
0        seal      982    988              99.39

Average Top-5 Accuracy: 99.39%
